In [12]:
%pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd

ROUND1_PATH = "round1.xlsx"
ROUND2_PATH = "round2.xlsx"

round1_df = pd.read_excel(ROUND1_PATH)
round2_df = pd.read_excel(ROUND2_PATH)

print("Round 1 shape:", round1_df.shape)
print("Round 2 shape:", round2_df.shape)

print("\nRound 1 columns:")
print(round1_df.columns.tolist())

print("\nRound 2 columns:")
print(round2_df.columns.tolist())

print("\nRound 1 preview:")
display(round1_df.head())

print("\nRound 2 preview:")
display(round2_df.head())

Round 1 shape: (125, 5)
Round 2 shape: (80, 5)

Round 1 columns:
['candidate_id', 'target_class_id', 'target_class', 'generated_text', 'human_decision']

Round 2 columns:
['candidate_id', 'target_class_id', 'target_class', 'generated_text', 'human_decision']

Round 1 preview:


,candidate_id,target_class_id,target_class,generated_text,human_decision
0,sexism_high_precision_v1_0500,3,Sexism,tu kutti ka bacha tha teri maa ki phudi mein k...,reject
1,sexism_high_precision_v1_0501,3,Sexism,sachai mai teri behn ka yaar chupay ga,reject
2,sexism_high_precision_v1_0502,3,Sexism,hijra gharwal nh ðŸ˜’,reject
3,sexism_high_precision_v1_0503,3,Sexism,aye nawaz sb charsi,reject
4,sexism_high_precision_v1_0504,3,Sexism,ye hi bharwe tera jism meri marzi tere maa ki ...,reject



Round 2 preview:


,candidate_id,target_class_id,target_class,generated_text,human_decision
0,round2_abusive_original_demo_v2_0900,0,Abusive/Offensive,jis din tu apna ghar ghar jata tu mar ja,reject
1,round2_abusive_original_demo_v2_0901,0,Abusive/Offensive,kisi aurat ka rona dekhna mujhe pata chala tha...,accept
2,round2_abusive_original_demo_v2_0902,0,Abusive/Offensive,agar apni maa behan ko aise hi mazeed nhi bolt...,reject
3,round2_abusive_original_demo_v2_0903,0,Abusive/Offensive,randi ka bacha,reject
4,round2_abusive_original_demo_v2_0904,0,Abusive/Offensive,ye dahi ke chote mc hain,reject


In [18]:
# ============================================================
# Clean Round 1 and Round 2 human annotations
# ============================================================

# Remove empty Excel "Unnamed" columns
round1_df = round1_df.loc[
    :, ~round1_df.columns.str.startswith("Unnamed")
].copy()

round2_df = round2_df.loc[
    :, ~round2_df.columns.str.startswith("Unnamed")
].copy()


# Standardise human decisions
for df in [round1_df, round2_df]:
    df["human_decision"] = (
        df["human_decision"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({
            "accept": "Accept",
            "reject": "Reject"
        })
    )


print("Round 1 shape after cleaning:", round1_df.shape)
print("Round 2 shape after cleaning:", round2_df.shape)

print("\nRound 1 columns:")
print(round1_df.columns.tolist())

print("\nRound 2 columns:")
print(round2_df.columns.tolist())

Round 1 shape after cleaning: (125, 5)
Round 2 shape after cleaning: (80, 5)

Round 1 columns:
['candidate_id', 'target_class_id', 'target_class', 'generated_text', 'human_decision']

Round 2 columns:
['candidate_id', 'target_class_id', 'target_class', 'generated_text', 'human_decision']


In [19]:
# ============================================================
# Check human annotation quality and distribution
# ============================================================

# Add round identifier
round1_df["round"] = "Round 1"
round2_df["round"] = "Round 2"

# Combine temporarily for diagnostics
diagnostic_df = pd.concat(
    [round1_df, round2_df],
    ignore_index=True,
    sort=False
)

print("Total annotated samples:", len(diagnostic_df))

# ------------------------------------------------------------
# 1. Human decision values
# ------------------------------------------------------------

print("\nHuman decision values:")
print(diagnostic_df["human_decision"].value_counts(dropna=False))

# ------------------------------------------------------------
# 2. Accept / Reject by target class
# ------------------------------------------------------------

print("\nAccept / Reject by target class:")

class_decisions = pd.crosstab(
    diagnostic_df["target_class"],
    diagnostic_df["human_decision"],
    margins=True
)

display(class_decisions)

# ------------------------------------------------------------
# 3. Distribution by round and class
# ------------------------------------------------------------

print("\nSamples by round and target class:")

round_class_distribution = pd.crosstab(
    diagnostic_df["round"],
    diagnostic_df["target_class"],
    margins=True
)

display(round_class_distribution)

# ------------------------------------------------------------
# 4. Missing human annotations
# ------------------------------------------------------------

missing_decisions = diagnostic_df[
    diagnostic_df["human_decision"].isna()
    | ~diagnostic_df["human_decision"].isin(
        ["Accept", "Reject"]
    )
]

print("\nMissing or invalid human decisions:",
      len(missing_decisions))

if len(missing_decisions) > 0:
    display(
        missing_decisions[
            [
                "candidate_id",
                "target_class",
                "generated_text",
                "human_decision"
            ]
        ]
    )

# ------------------------------------------------------------
# 5. Duplicate candidate IDs
# ------------------------------------------------------------

duplicate_ids = diagnostic_df[
    diagnostic_df["candidate_id"].duplicated(
        keep=False
    )
]

print("\nDuplicate candidate IDs:",
      len(duplicate_ids))

if len(duplicate_ids) > 0:
    display(duplicate_ids)

# ------------------------------------------------------------
# 6. Duplicate generated texts
# ------------------------------------------------------------

duplicate_texts = diagnostic_df[
    diagnostic_df["generated_text"].duplicated(
        keep=False
    )
].sort_values("generated_text")

print("\nRows involved in duplicate generated texts:",
      len(duplicate_texts))

if len(duplicate_texts) > 0:
    display(
        duplicate_texts[
            [
                "candidate_id",
                "round",
                "target_class",
                "generated_text",
                "human_decision"
            ]
        ]
    )

Total annotated samples: 205

Human decision values:
human_decision
Reject    140
Accept     65
Name: count, dtype: int64

Accept / Reject by target class:


human_decision,Accept,Reject,All
target_class,,,
Abusive/Offensive,18,22,40
Profane,13,27,40
Religious Hate,16,49,65
Sexism,18,42,60
All,65,140,205



Samples by round and target class:


target_class,Abusive/Offensive,Profane,Religious Hate,Sexism,All
round,,,,,
Round 1,20,20,45,40,125
Round 2,20,20,20,20,80
All,40,40,65,60,205



Missing or invalid human decisions: 0

Duplicate candidate IDs: 0

Rows involved in duplicate generated texts: 2


,candidate_id,round,target_class,generated_text,human_decision
128,round2_abusive_original_demo_v2_0903,Round 2,Abusive/Offensive,randi ka bacha,Reject
138,round2_abusive_original_demo_v2_0913,Round 2,Abusive/Offensive,randi ka bacha,Reject


In [20]:
# ============================================================
# Remove exact duplicate generated texts
# ============================================================

clean_df = diagnostic_df.drop_duplicates(
    subset=["generated_text"],
    keep="first"
).reset_index(drop=True)

print("Samples after duplicate removal:", len(clean_df))
print(
    "Remaining duplicate generated texts:",
    clean_df["generated_text"].duplicated().sum()
)

display(
    pd.crosstab(
        clean_df["target_class"],
        clean_df["human_decision"],
        margins=True
    )
)

Samples after duplicate removal: 204
Remaining duplicate generated texts: 0


human_decision,Accept,Reject,All
target_class,,,
Abusive/Offensive,18,21,39
Profane,13,27,40
Religious Hate,16,49,65
Sexism,18,42,60
All,65,139,204


In [21]:
# ============================================================
# Create frozen balanced judge-validation set
# 10 Accept + 10 Reject per target class
# ============================================================

FINAL_SEED = 42

selected_parts = []

target_classes = [
    "Abusive/Offensive",
    "Religious Hate",
    "Sexism",
    "Profane",
]

for target_class in target_classes:

    class_df = clean_df[
        clean_df["target_class"] == target_class
    ]

    accepts = class_df[
        class_df["human_decision"] == "Accept"
    ].sample(
        n=10,
        random_state=FINAL_SEED
    )

    rejects = class_df[
        class_df["human_decision"] == "Reject"
    ].sample(
        n=10,
        random_state=FINAL_SEED
    )

    selected_parts.extend(
        [accepts, rejects]
    )


final_judge_validation_df = pd.concat(
    selected_parts,
    ignore_index=True
)

# Randomise final order
final_judge_validation_df = (
    final_judge_validation_df
    .sample(
        frac=1,
        random_state=FINAL_SEED
    )
    .reset_index(drop=True)
)


print(
    "Final judge-validation samples:",
    len(final_judge_validation_df)
)

display(
    pd.crosstab(
        final_judge_validation_df["target_class"],
        final_judge_validation_df["human_decision"],
        margins=True
    )
)

Final judge-validation samples: 80


human_decision,Accept,Reject,All
target_class,,,
Abusive/Offensive,10,10,20
Profane,10,10,20
Religious Hate,10,10,20
Sexism,10,10,20
All,40,40,80


In [22]:
# ============================================================
# Save frozen reference set and blinded judge input set
# ============================================================

FINAL_REFERENCE_PATH = (
    "judge_validation_final_reference.csv"
)

FINAL_JUDGE_INPUT_PATH = (
    "judge_validation_final_input.csv"
)


# Full human-reference file
final_judge_validation_df.to_csv(
    FINAL_REFERENCE_PATH,
    index=False,
    encoding="utf-8"
)


# Blinded file for Qwen / Prometheus
judge_input_columns = [
    "candidate_id",
    "target_class_id",
    "target_class",
    "generated_text",
]

final_judge_validation_df[
    judge_input_columns
].to_csv(
    FINAL_JUDGE_INPUT_PATH,
    index=False,
    encoding="utf-8"
)


print("Saved reference set:")
print(FINAL_REFERENCE_PATH)

print("\nSaved blinded judge input set:")
print(FINAL_JUDGE_INPUT_PATH)

Saved reference set:
judge_validation_final_reference.csv

Saved blinded judge input set:
judge_validation_final_input.csv
